# Ứng dụng học máy và học sâu trong phân cụm giao dịch chuyển tiền IBFT
## 01. Chuẩn bị và xử lý dữ liệu

Đơn vị xử lý là **một giao dịch**. Notebook đọc 10 file Excel có sẵn trong
`data/`, tạo 11 đặc trưng và xuất dữ liệu dùng chung cho ba mô hình.
Theo yêu cầu hiện tại, không tích hợp bước sinh lại dữ liệu.

Lịch sử chỉ dùng giao dịch **trước thời điểm hiện tại**; 7 ngày đầu làm
lịch sử nền. Chia tập theo thời gian 70% / 15% / 15%, giữ các giao dịch
cùng thời điểm trong cùng một tập. Điền thiếu và chuẩn hóa chỉ học từ train.

Chọn kernel Python của `.venv`, rồi **Restart Kernel → Run All**.
Chỉ bật `INSTALL_DEPENDENCIES` khi cần cài môi trường; mặc định dùng thư viện đã cài.
`requirements.txt` đã có `matplotlib`; `torch` sẽ được bổ sung ở bước Autoencoder.

In [ ]:
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    from pathlib import Path

    requirements_path = next(
        (folder / "requirements.txt"
         for folder in [Path.cwd(), *Path.cwd().parents]
         if (folder / "requirements.txt").is_file()),
        None,
    )
    if requirements_path is None:
        raise FileNotFoundError("Không tìm thấy requirements.txt của dự án.")
    # Bao gồm pandas, numpy, openpyxl, pyarrow, scikit-learn, joblib, matplotlib.
    %pip install -r "{requirements_path}"
else:
    print("Dùng môi trường hiện tại; bật INSTALL_DEPENDENCIES nếu cần cài thư viện.")

### 1. Thiết lập và cấu hình

In [ ]:
from pathlib import Path
import json
import sys
import hashlib
from datetime import datetime, timezone
from importlib.metadata import version

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import PercentFormatter

from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def find_project_root(start_path=None):
    """Tìm thư mục dự án chứa thư mục data/."""
    start_path = Path.cwd() if start_path is None else Path(start_path)
    start_path = start_path.resolve()

    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data").is_dir():
            return candidate

    raise FileNotFoundError(
        "Không tìm thấy thư mục gốc dự án có chứa data/. "
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
PREPARED_DIR = OUTPUT_DIR / "prepared"
FIGURES_DIR = PREPARED_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})


print("Python:", sys.version.split()[0])
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

In [ ]:
# Thông tin file Excel
SHEET_NAME = "IBFT"
INPUT_PATTERN = "IBFT_SYNTHETIC_MAIN_*.xlsx"
EXPECTED_FILE_COUNT = 10

# Mã phản hồi 0 nghĩa là giao dịch thành công
SUCCESS_RESPONSE_CODE = 0

# Tỷ lệ chia dữ liệu theo thời gian
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

# Tên những cột được sử dụng nhiều lần
TIME_COL = "transaction_datetime"
AMOUNT_COL = "transactionamount"
SENDER_COL = "accountsourcenumber"
RECEIVER_COL = "accountdestnumber"

print("\nCấu hình chia dữ liệu:")
print(f"  Train:      {TRAIN_RATIO:.0%}")
print(f"  Validation: {VALIDATION_RATIO:.0%}")
print(f"  Test:       {TEST_RATIO:.0%}")

assert all(0 < ratio < 1 for ratio in [TRAIN_RATIO, VALIDATION_RATIO, TEST_RATIO])
assert np.isclose(TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO, 1.0)

## 2. Đọc và gộp các file Excel
Đọc 10 file Excel trong thư mục `data/`, sau đó gộp thành một bảng.

Mỗi giao dịch được bổ sung:

- `source_file`: tên file nguồn.
- `source_row`: số dòng trong file Excel.

In [ ]:
excel_files = sorted(DATA_DIR.glob(INPUT_PATTERN))

if len(excel_files) != EXPECTED_FILE_COUNT:
    raise ValueError(
        f"Cần {EXPECTED_FILE_COUNT} file, "
        f"nhưng tìm thấy {len(excel_files)} file."
    )

dataframes = []
input_file_info = []

for file_path in excel_files:
    df = pd.read_excel(
        file_path,
        sheet_name=SHEET_NAME,
        dtype={
            SENDER_COL: "string",
            RECEIVER_COL: "string",
            "referencenumber": "string",
        },
    )

    required_columns = {
        SENDER_COL, RECEIVER_COL, AMOUNT_COL, "referencenumber",
        "responsecode", "short_filename", "timelocaltranx", "datelocaltranx",
    }
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"{file_path.name} thiếu cột: {sorted(missing_columns)}")

    input_file_info.append({
        "file": file_path.name,
        "row_count": len(df),
        "sha256": hashlib.sha256(file_path.read_bytes()).hexdigest(),
    })
    df["source_file"] = file_path.name
    df["source_row"] = np.arange(2, len(df) + 2) # cột thể hiện số dòng trong file

    dataframes.append(df)

data = pd.concat(dataframes, ignore_index=True)

print("Số file:", len(excel_files))
print("Số giao dịch:", len(data))
print("Số cột:", len(data.columns))

## 3. Xem dữ liệu

Chỉ hiển thị một số cột quan trọng để tránh bảng quá rộng.

In [ ]:
preview_columns = [
    "referencenumber",
    AMOUNT_COL,
    "datelocaltranx",
    "timelocaltranx",
    SENDER_COL,
    RECEIVER_COL,
    "responsecode",
    "source_file",
]

display(data[preview_columns].head())

## 4. Kiểm tra dữ liệu

Kiểm tra:

- Số lượng giá trị thiếu.
- Dòng bị trùng.
- Mã tham chiếu giao dịch bị trùng.
- Số lượng từng loại mã phản hồi.

In [ ]:
missing_values = data.isna().sum()
missing_values = missing_values[missing_values > 0]

print("Số dòng trùng:", data.duplicated().sum())

print("Số referencenumber bị trùng:",
    data["referencenumber"].duplicated().sum(),
)

print("\nSố lượng theo responsecode (0 là thành công):")
display(data["responsecode"].value_counts(dropna=False))

print("Các cột có dữ liệu thiếu:")
display(missing_values.to_frame("missing_count"))

In [ ]:
# Cột địa chỉ không dùng để tạo đặc trưng hoặc tra cứu.
data = data.drop(columns=["accountsourceaddress"], errors="ignore")
print("Số cột:", len(data.columns))

## 5. Tạo thời điểm giao dịch

Ngày được lấy từ `short_filename`.

Giờ được lấy từ `timelocaltranx`.

Hai trường này được ghép thành `transaction_datetime`.

In [ ]:
# Số ngày dùng để đo mức độ hoạt động gần đây
HISTORY_WINDOW_DAYS = 7

# tìm thời gian ở cột short_filename
date_text = data["short_filename"].str.extract(
    r"(\d{4}-\d{2}-\d{2})",
    expand=False,
)

time_text = (
    pd.to_numeric(
        data["timelocaltranx"],
        errors="coerce",
    )
    .astype("Int64")
    .astype("string")
    .str.zfill(6)
)
# tạo thành cột có giờ + ngày của 1 giao dịch
data[TIME_COL] = pd.to_datetime(
    date_text + " " + time_text,
    format="%Y-%m-%d %H%M%S",
    errors="coerce",
)

data[AMOUNT_COL] = pd.to_numeric(
    data[AMOUNT_COL],
    errors="coerce",
)

data["responsecode"] = pd.to_numeric(
    data["responsecode"],
    errors="coerce",
)

data["transaction_id"] = (
    data["source_file"].str.replace(
        ".xlsx",
        "",
        regex=False,
    )
    + ":"
    + data["source_row"].astype(str)
)

print("Thời gian bắt đầu (của ~45000 giao dịch):", data[TIME_COL].min())
print("Thời gian kết thúc (của ~45000 giao dịch):", data[TIME_COL].max())

## 6. Lọc dữ liệu

Chỉ giữ giao dịch:

- Thành công: `responsecode = 0`.
- Có thời gian giao dịch.
- Có tài khoản gửi và tài khoản nhận.
- Số tiền lớn hơn 0 hoặc đang bị thiếu.

Số tiền bị thiếu chưa được điền ở bước này. Việc điền thiếu sẽ được học từ tập train.

In [ ]:
row_count_before = len(data)


clean_data = data[
    data["responsecode"].eq(SUCCESS_RESPONSE_CODE)
    & data[TIME_COL].notna()
    & data[SENDER_COL].notna()
    & data[RECEIVER_COL].notna()
    & (
        data[AMOUNT_COL].isna()
        | data[AMOUNT_COL].gt(0)
    )
].copy()

duplicate_reference = (
    clean_data["referencenumber"].notna()
    & clean_data["referencenumber"].duplicated()
)

clean_data = clean_data[
    ~duplicate_reference
].copy()

clean_data = (
    clean_data
    .sort_values(
        [TIME_COL, "transaction_id"]
    )
    .reset_index(drop=True)
)

print("Trước khi lọc:", row_count_before)
print("Sau khi lọc:", len(clean_data))
print("Số dòng bị loại:", row_count_before - len(clean_data))
print(
    "Số dòng còn thiếu số tiền:",
    clean_data[AMOUNT_COL].isna().sum(),
)

filtered_row_count = len(clean_data)
if clean_data.empty:
    raise ValueError("Không còn giao dịch hợp lệ sau bước lọc.")

## 7. Lịch sử tài khoản gửi

Với mỗi giao dịch tại thời điểm `t`, tính:

- Số giao dịch của người gửi trong khoảng `[t − 7 ngày, t)`.
- Thời điểm giao dịch gần nhất trong toàn bộ quá khứ.

Các giao dịch cùng thời điểm không được tính là lịch sử của nhau.

In [ ]:
sender_history = (
    clean_data
    .groupby([SENDER_COL, TIME_COL])
    .size()
    .reset_index(name="sender_tx_at_time")
    .sort_values([SENDER_COL, TIME_COL])
    .reset_index(drop=True)
)

# Lần giao dịch gần nhất trong toàn bộ quá khứ
sender_history["sender_previous_time"] = (
    sender_history
    .groupby(SENDER_COL)[TIME_COL]
    .shift(1)
)

# Số giao dịch trong 7 ngày trước thời điểm hiện tại, các giao dịch cùng thời điểm không được tính là lịch sử của nhau.
sender_count_7d = (
    sender_history
    .groupby(SENDER_COL, sort=False)
    .rolling(
        window=f"{HISTORY_WINDOW_DAYS}D",
        on=TIME_COL,
        closed="left",
    )["sender_tx_at_time"]
    .sum()
    .rename("sender_tx_count_7d")
    .reset_index()
)

sender_history = sender_history.merge(
    sender_count_7d,
    on=[SENDER_COL, TIME_COL],
    how="left",
    validate="one_to_one",
)

sender_history["sender_tx_count_7d"] = (
    sender_history["sender_tx_count_7d"]
    .fillna(0)
    .astype(int)
)

clean_data = clean_data.merge(
    sender_history[
        [
            SENDER_COL,
            TIME_COL,
            "sender_previous_time",
            "sender_tx_count_7d",
        ]
    ],
    on=[SENDER_COL, TIME_COL],
    how="left",
    validate="many_to_one",
)

## 8. Lịch sử cặp gửi – nhận

Với mỗi giao dịch tại thời điểm `t`, tính:

- Số lần người gửi chuyển cho người nhận trong khoảng `[t − 7 ngày, t)`.
- Người gửi đã từng chuyển cho người nhận này trong toàn bộ quá khứ hay chưa.

In [ ]:
pair_history = (
    clean_data
    .groupby(
        [SENDER_COL, RECEIVER_COL, TIME_COL]
    )
    .size()
    .reset_index(name="pair_tx_at_time")
    .sort_values(
        [SENDER_COL, RECEIVER_COL, TIME_COL]
    )
    .reset_index(drop=True)
)

# Người gửi đã từng chuyển cho người nhận này chưa
pair_history["is_new_receiver"] = (
    pair_history
    .groupby([SENDER_COL, RECEIVER_COL])
    .cumcount()
    .eq(0)
    .astype(int)
)

# Số lần chuyển cho người nhận trong 7 ngày trước
pair_count_7d = (
    pair_history
    .groupby(
        [SENDER_COL, RECEIVER_COL],
        sort=False,
    )
    .rolling(
        window=f"{HISTORY_WINDOW_DAYS}D",
        on=TIME_COL,
        closed="left",
    )["pair_tx_at_time"]
    .sum()
    .rename("pair_tx_count_7d")
    .reset_index()
)

pair_history = pair_history.merge(
    pair_count_7d,
    on=[
        SENDER_COL,
        RECEIVER_COL,
        TIME_COL,
    ],
    how="left",
    validate="one_to_one",
)

pair_history["pair_tx_count_7d"] = (
    pair_history["pair_tx_count_7d"]
    .fillna(0)
    .astype(int)
)

clean_data = clean_data.merge(
    pair_history[
        [
            SENDER_COL,
            RECEIVER_COL,
            TIME_COL,
            "pair_tx_count_7d",
            "is_new_receiver",
        ]
    ],
    on=[
        SENDER_COL,
        RECEIVER_COL,
        TIME_COL,
    ],
    how="left",
    validate="many_to_one",
)

## 9. Tạo đặc trưng lịch sử

Các đặc trưng gồm:

- Khoảng cách với lần chuyển trước trong toàn bộ quá khứ.
- Giao dịch đầu tiên của người gửi.
- Người nhận mới trong toàn bộ quá khứ.
- Tỷ trọng chuyển cho người nhận trong cửa sổ 7 ngày.

In [ ]:
# Khoảng cách với lần chuyển trước trong toàn bộ quá khứ
clean_data["sender_recency_days"] = (
    clean_data[TIME_COL]
    - clean_data["sender_previous_time"]
).dt.total_seconds() / 86_400

# Có phải giao dịch đầu tiên của người gửi không
clean_data["is_first_sender_tx"] = (
    clean_data["sender_previous_time"]
    .isna()
    .astype(int)
)

# Tỷ trọng chuyển cho người nhận trong cùng cửa sổ 7 ngày
clean_data["receiver_share_7d"] = np.where(
    clean_data["sender_tx_count_7d"].gt(0),
    (
        clean_data["pair_tx_count_7d"]
        / clean_data["sender_tx_count_7d"]
    ),
    0.0,
)

print(
    clean_data[
        [
            TIME_COL,
            "sender_tx_count_7d",
            "pair_tx_count_7d",
            "receiver_share_7d",
            "sender_recency_days",
            "is_first_sender_tx",
            "is_new_receiver",
        ]
    ].head()
)

## 10. Giữ 7 ngày đầu làm lịch sử nền

Các giao dịch trong 7 ngày đầu chỉ được dùng để tính lịch sử.

Chúng không được đưa vào train, validation hoặc test.

In [ ]:
history_start_time = clean_data[TIME_COL].min()

model_start_time = (
    history_start_time
    + pd.Timedelta(days=HISTORY_WINDOW_DAYS)
)

history_row_count = (
    clean_data[TIME_COL]
    .lt(model_start_time)
    .sum()
)

# Giữ hai cờ trên toàn bộ dữ liệu đã lọc để quan sát cả 7 ngày nền.
history_diagnostics = clean_data[
    [TIME_COL, "is_first_sender_tx", "is_new_receiver"]
].copy()

clean_data = (
    clean_data[
        clean_data[TIME_COL].ge(model_start_time)
    ]
    .sort_values([TIME_COL, "transaction_id"])
    .reset_index(drop=True)
)

print("Bắt đầu dữ liệu gốc:", history_start_time)
print("Bắt đầu dữ liệu mô hình:", model_start_time)
print("Số giao dịch làm lịch sử nền:", history_row_count)
print("Số giao dịch đưa vào mô hình:", len(clean_data))

### 10.1. Tỷ lệ người nhận mới và giao dịch đầu của người gửi theo tuần

Mỗi tuần là một khoảng 7 ngày liên tiếp tính từ đầu dữ liệu, trùng với
ranh giới 7 ngày nền. Mẫu số là **số giao dịch đã lọc trong tuần**,
không phải số tài khoản; cả các giao dịch cùng thời điểm đều được tính.
Tuần cuối có thể chưa đủ 7 ngày. Bảng bên dưới ghi số giao dịch và số ngày quan sát.

Hai biểu đồ dùng thang tỷ lệ riêng để nhìn được thay đổi của cờ rất hiếm.
Các tỷ lệ chỉ phục vụ kiểm tra dữ liệu, không dùng để học tham số xử lý.

In [ ]:
week_number = (
    (history_diagnostics[TIME_COL] - history_start_time)
    // pd.Timedelta(days=7)
).astype(int)
history_diagnostics["week_start"] = (
    history_start_time + pd.to_timedelta(week_number * 7, unit="D")
)
weekly_binary_rates = (
    history_diagnostics.groupby("week_start", sort=True)
    .agg(
        row_count=(TIME_COL, "size"),
        observed_start=(TIME_COL, "min"),
        observed_end=(TIME_COL, "max"),
        new_receiver_count=("is_new_receiver", "sum"),
        first_sender_tx_count=("is_first_sender_tx", "sum"),
    )
    .reset_index()
)
weekly_binary_rates["is_new_receiver_rate"] = (
    weekly_binary_rates["new_receiver_count"] / weekly_binary_rates["row_count"]
)
weekly_binary_rates["is_first_sender_tx_rate"] = (
    weekly_binary_rates["first_sender_tx_count"] / weekly_binary_rates["row_count"]
)
weekly_binary_rates["observed_days"] = (
    weekly_binary_rates["observed_end"].dt.normalize()
    - weekly_binary_rates["observed_start"].dt.normalize()
).dt.days + 1
weekly_binary_rates["history_only"] = (
    weekly_binary_rates["week_start"] + pd.Timedelta(days=7)
    <= model_start_time
)
assert weekly_binary_rates["row_count"].sum() == filtered_row_count
display(weekly_binary_rates[[
    "week_start", "observed_days", "row_count", "history_only",
    "is_new_receiver_rate", "is_first_sender_tx_rate",
]].style.format({
    "week_start": lambda value: value.strftime("%d/%m/%Y"),
    "is_new_receiver_rate": "{:.2%}",
    "is_first_sender_tx_rate": "{:.2%}",
}))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
rate_specs = [
    ("is_new_receiver_rate", "Người nhận mới trong lịch sử quan sát", "#2457A6", "o"),
    ("is_first_sender_tx_rate", "Giao dịch đầu của người gửi trong lịch sử quan sát", "#B27619", "s"),
]
for ax, (column, title, color, marker) in zip(axes, rate_specs):
    ax.plot(weekly_binary_rates["week_start"], weekly_binary_rates[column],
            color=color, marker=marker, linewidth=2)
    ax.axvspan(history_start_time, model_start_time, color="#D8D8D8", alpha=0.45)
    ax.axvline(model_start_time, color="#555555", linestyle="--", linewidth=1)
    ax.set_ylim(0, min(1.0, max(0.01, weekly_binary_rates[column].max() * 1.20)))
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=1))
    ax.set_title(title, loc="left")
    ax.set_ylabel("Tỷ lệ giao dịch (%)")
    ax.grid(axis="y", alpha=0.25)
axes[0].text(0.99, 0.94, "Vùng xám: 7 ngày lịch sử nền", ha="right", va="top",
             transform=axes[0].transAxes, color="#555555")
axes[-1].set_xticks(weekly_binary_rates["week_start"])
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
axes[-1].set_xlabel("Ngày bắt đầu tuần năm 2026 (tuần cuối có thể chưa đủ 7 ngày)")
fig.suptitle("Tỷ lệ hai đặc trưng nhị phân theo tuần — thang dọc riêng cho từng biểu đồ")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "weekly_binary_rates.png", dpi=150, bbox_inches="tight")
plt.show()

Nhận xét từ dữ liệu đang chạy:

Tỷ lệ người nhận mới: 68.64% ở tuần nền, 47.78% ở tuần mô hình đầu, và 31.78% ở tuần cuối.
Tỷ lệ giao dịch đầu của người gửi: 9.60%, 0.59%, và 0.00% trong ba tuần tương ứng.
Trong dữ liệu mô hình, cờ giao dịch đầu bằng 1 ở 40/40,042 giao dịch (0.100%).
Tuần cuối có 4 ngày quan sát; cần lưu ý khi so sánh với các tuần đủ 7 ngày.

**Giới hạn diễn giải:** “Đầu tiên” và “mới” chỉ có nghĩa là chưa từng xuất hiện
trong phần lịch sử quan sát được từ đầu bộ dữ liệu, không khẳng định hành vi
trước đó của tài khoản. Bỏ 7 ngày đầu giúp giảm ảnh hưởng của lịch sử chưa đầy đủ
nhưng không bảo đảm loại bỏ hoàn toàn ảnh hưởng này. Giữ nguyên hai cờ 0/1 theo
đặc tả; khi mô tả cụm cần đối chiếu thời gian và mức độ hiếm của cờ.

## 11. Tạo đặc trưng số tiền và thời gian

Sử dụng `log1p` cho số tiền, số lần và khoảng cách thời gian để giảm độ lệch lớn.

Giờ trong ngày và thứ trong tuần được chuyển thành sin/cos vì chúng có tính tuần hoàn.

In [ ]:
seconds_of_day = (
    clean_data[TIME_COL].dt.hour * 3_600
    + clean_data[TIME_COL].dt.minute * 60
    + clean_data[TIME_COL].dt.second
)

day_of_week = clean_data[TIME_COL].dt.dayofweek

clean_data["log_amount"] = np.log1p(
    clean_data[AMOUNT_COL]
)

clean_data["hour_sin"] = np.sin(
    2 * np.pi * seconds_of_day / 86_400
)

clean_data["hour_cos"] = np.cos(
    2 * np.pi * seconds_of_day / 86_400
)

clean_data["weekday_sin"] = np.sin(
    2 * np.pi * day_of_week / 7
)

clean_data["weekday_cos"] = np.cos(
    2 * np.pi * day_of_week / 7
)

clean_data["log_sender_recency_days"] = np.log1p(
    clean_data["sender_recency_days"]
)

clean_data["log_sender_tx_count_7d"] = np.log1p(
    clean_data["sender_tx_count_7d"]
)

clean_data["log_pair_tx_count_7d"] = np.log1p(
    clean_data["pair_tx_count_7d"]
)

### 11.1. Phân phối số tiền trước và sau biến đổi log

Biểu đồ giúp quan sát ảnh hưởng của biến đổi log đối với phân phối số tiền.

In [ ]:
amount_values = clean_data[
    AMOUNT_COL
].dropna()

amount_p99 = amount_values.quantile(0.99)

amount_for_plot = (
    amount_values
    .clip(upper=amount_p99)
    / 1_000_000
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4.5),
)

axes[0].hist(
    amount_for_plot,
    bins=50,
    color="#2457A6",
    edgecolor="white",
)

axes[0].set_title(
    "Số tiền giao dịch gốc\n(hiển thị đến P99)"
)
axes[0].set_xlabel("Số tiền (triệu đồng)")
axes[0].set_ylabel("Số giao dịch")
axes[0].grid(alpha=0.25)

axes[1].hist(
    clean_data["log_amount"].dropna(),
    bins=50,
    color="#4C956C",
    edgecolor="white",
)

axes[1].set_title(
    "Số tiền sau biến đổi log1p"
)
axes[1].set_xlabel("log(1 + số tiền)")
axes[1].set_ylabel("Số giao dịch")
axes[1].grid(alpha=0.25)

fig.suptitle(
    "Phân phối số tiền trước và sau biến đổi log",
    fontsize=14,
)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "amount_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### 11.2. Phân phối đặc trưng hoạt động gần đây

Các biểu đồ mô tả hoạt động của người gửi và mối quan hệ với người nhận
tại thời điểm mỗi giao dịch được thực hiện.

In [ ]:
activity_features = [
    (
        "sender_tx_count_7d",
        "Số giao dịch của người gửi\ntrong 7 ngày",
        "Số giao dịch",
        "#2457A6",
    ),
    (
        "pair_tx_count_7d",
        "Số lần chuyển cho người nhận\ntrong 7 ngày",
        "Số giao dịch",
        "#4C956C",
    ),
    (
        "sender_recency_days",
        "Khoảng cách với lần chuyển trước",
        "Số ngày",
        "#D4A72C",
    ),
    (
        "receiver_share_7d",
        "Tỷ trọng chuyển cho người nhận\ntrong 7 ngày",
        "Tỷ trọng",
        "#E07A5F",
    ),
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 8),
)

for ax, (
    column,
    title,
    xlabel,
    color,
) in zip(
    axes.flatten(),
    activity_features,
):
    values = clean_data[
        column
    ].dropna()

    # Với các biến có giá trị lớn,
    # chỉ giới hạn hình tại P99
    if column != "receiver_share_7d":
        upper_limit = values.quantile(0.99)

        values = values.clip(
            upper=upper_limit
        )

        title += "\n(hiển thị đến P99)"

    ax.hist(
        values,
        bins=40,
        color=color,
        edgecolor="white",
    )

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Số giao dịch")
    ax.grid(alpha=0.25)

fig.suptitle(
    "Phân phối các đặc trưng hoạt động giao dịch",
    fontsize=14,
)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "activity_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Chọn đặc trưng

Số tài khoản gửi và nhận không nằm trong danh sách đặc trưng.

Chúng chỉ được sử dụng để tính lịch sử giao dịch.

In [ ]:
CONTINUOUS_FEATURES = [
    "log_amount",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "log_sender_recency_days",
    "log_sender_tx_count_7d",
    "log_pair_tx_count_7d",
    "receiver_share_7d",
]

BINARY_FEATURES = [
    "is_first_sender_tx",
    "is_new_receiver",
]

FEATURE_COLUMNS = (
    CONTINUOUS_FEATURES
    + BINARY_FEATURES
)

display(
    clean_data[FEATURE_COLUMNS].head()
)

## 13. Chia train, validation và test

Dữ liệu không được xáo trộn.

- 70% đầu: train.
- 15% tiếp theo: validation.
- 15% cuối: test.

Các giao dịch cùng thời điểm được giữ chung một tập, nên tỷ lệ thực tế có thể lệch nhẹ.
Lịch sử ở validation/test được cập nhật từ những giao dịch đã xảy ra trước đó;
đây là lịch sử quan sát, không phải thao tác fit lại bộ chuẩn hóa.

In [ ]:
number_of_rows = len(clean_data)

train_position = int(
    number_of_rows * TRAIN_RATIO
)

validation_position = int(
    number_of_rows
    * (TRAIN_RATIO + VALIDATION_RATIO)
)

if not 0 < train_position < validation_position < number_of_rows:
    raise ValueError("Không đủ giao dịch để chia ba tập theo tỷ lệ đã chọn.")

train_cutoff = clean_data.iloc[
    train_position
][TIME_COL]

validation_cutoff = clean_data.iloc[
    validation_position
][TIME_COL]

clean_data["split"] = np.select(
    [
        clean_data[TIME_COL] < train_cutoff,
        clean_data[TIME_COL] < validation_cutoff,
    ],
    [
        "train",
        "validation",
    ],
    default="test",
)

split_summary = (
    clean_data
    .groupby("split")
    .agg(
        row_count=("transaction_id", "size"),
        start_time=(TIME_COL, "min"),
        end_time=(TIME_COL, "max"),
    )
    .reindex(["train", "validation", "test"])
)

display(split_summary)

if split_summary["row_count"].isna().any() or split_summary["row_count"].le(0).any():
    raise ValueError("Không chia được ba tập không rỗng theo thời gian; kiểm tra dữ liệu trùng thời điểm.")

## 14. Xử lý thiếu và chuẩn hóa

Bộ xử lý chỉ được học từ tập train.

- Biến liên tục: điền median và chuẩn hóa.
- Biến nhị phân: điền giá trị xuất hiện nhiều nhất và giữ nguyên 0/1.

Validation và test chỉ được biến đổi bằng tham số đã học từ train.

In [ ]:
continuous_processor = Pipeline([
    (
        "fill_missing",
        SimpleImputer(strategy="median"),
    ),
    (
        "standardize",
        StandardScaler(),
    ),
])

preprocessor = ColumnTransformer([
    (
        "continuous",
        continuous_processor,
        CONTINUOUS_FEATURES,
    ),
    (
        "binary",
        SimpleImputer(strategy="most_frequent"),
        BINARY_FEATURES,
    ),
])

train_rows = clean_data["split"].eq("train")

empty_train_features = clean_data.loc[train_rows, FEATURE_COLUMNS].isna().all()
if empty_train_features.any():
    raise ValueError(f"Đặc trưng thiếu toàn bộ trên train: {empty_train_features[empty_train_features].index.tolist()}")

preprocessor.fit(
    clean_data.loc[
        train_rows,
        FEATURE_COLUMNS,
    ]
)

processed_data = preprocessor.transform(
    clean_data[FEATURE_COLUMNS]
)

print("Kích thước dữ liệu sau xử lý:", processed_data.shape)
print(
    "Còn NaN hoặc infinity:",
    not np.isfinite(processed_data).all(),
)

## 15. Tạo bảng đầu vào mô hình

Bảng đầu vào chỉ chứa:

- Mã giao dịch.
- Thời gian.
- Tên tập dữ liệu.
- Các đặc trưng đã xử lý.

Bảng này không chứa số tài khoản.

In [ ]:
PROCESSED_FEATURES = [
    f"x_{column}"
    for column in FEATURE_COLUMNS
]

processed_dataframe = pd.DataFrame(
    processed_data,
    columns=PROCESSED_FEATURES,
    index=clean_data.index,
)

model_input = pd.concat(
    [
        clean_data[
            [
                "transaction_id",
                TIME_COL,
                "split",
            ]
        ],
        processed_dataframe,
    ],
    axis=1,
)

display(model_input.head())

## 16. Kiểm tra trước khi lưu

In [ ]:
assert model_input["transaction_id"].is_unique

assert SENDER_COL not in model_input.columns
assert RECEIVER_COL not in model_input.columns

assert np.isfinite(
    model_input[PROCESSED_FEATURES].to_numpy()
).all()

assert set(model_input["split"]) == {
    "train",
    "validation",
    "test",
}

assert (
    clean_data["pair_tx_count_7d"]
    <= clean_data["sender_tx_count_7d"]
).all()

assert (
    clean_data["receiver_share_7d"]
    .between(0, 1)
    .all()
)

assert clean_data[TIME_COL].min() >= model_start_time

print("Kiểm tra hoàn tất.")
print("Số giao dịch:", len(model_input))
print("Số đặc trưng:", len(PROCESSED_FEATURES))

assert len(PROCESSED_FEATURES) == 11
assert processed_data.shape == (len(clean_data), len(PROCESSED_FEATURES))
assert model_input.columns.tolist() == ["transaction_id", TIME_COL, "split", *PROCESSED_FEATURES]
assert model_input[TIME_COL].is_monotonic_increasing
assert model_input.groupby(TIME_COL)["split"].nunique().eq(1).all()
assert split_summary.loc["train", "end_time"] < split_summary.loc["validation", "start_time"]
assert split_summary.loc["validation", "end_time"] < split_summary.loc["test", "start_time"]
assert int(split_summary["row_count"].sum()) == len(model_input)
assert len(model_input) + int(history_row_count) == filtered_row_count
assert model_input[[f"x_{name}" for name in BINARY_FEATURES]].isin([0, 1]).all().all()
assert (
    clean_data["sender_previous_time"].isna()
    | clean_data["sender_previous_time"].lt(clean_data[TIME_COL])
).all()
assert np.array_equal(
    model_input[[f"x_{name}" for name in BINARY_FEATURES]].to_numpy(),
    clean_data[BINARY_FEATURES].to_numpy(),
)
print("Đã xác nhận thứ tự thời gian, 11 đặc trưng và hai cờ giữ nguyên 0/1.")


## 17. Lưu dữ liệu đã chuẩn bị

Lưu:

- Dữ liệu đầu vào mô hình.
- Đặc trưng trước chuẩn hóa.
- Bảng tra cứu giao dịch.
- Bộ xử lý thiếu và chuẩn hóa.
- Thông tin chuẩn bị dữ liệu.

Chỉ ghi các bảng sau khi các kiểm tra nhất quán đã đạt. Biểu đồ được lưu trong
`output/prepared/figures/`; bảng tỷ lệ tuần là `weekly_binary_rates.csv`.

In [ ]:
PREPARED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

model_input.to_parquet(
    PREPARED_DIR / "model_input.parquet",
    index=False,
)

clean_data[
    [
        "transaction_id",
        TIME_COL,
        "split",
        *FEATURE_COLUMNS,
    ]
].to_parquet(
    PREPARED_DIR / "features_unscaled.parquet",
    index=False,
)

clean_data[
    [
        "transaction_id",
        "source_file",
        "source_row",
        "referencenumber",
        TIME_COL,
        AMOUNT_COL,
        SENDER_COL,
        RECEIVER_COL,
        "responsecode",
    ]
].to_parquet(
    PREPARED_DIR / "transaction_lookup.parquet",
    index=False,
)

joblib.dump(
    preprocessor,
    PREPARED_DIR / "preprocessor.joblib",
)

preparation_info = {
    "row_count": len(clean_data),
    "raw_row_count": row_count_before,
    "filtered_row_count": filtered_row_count,
    "excluded_row_count": row_count_before - filtered_row_count,
    "input_file_count": len(excel_files),
    "input_files": input_file_info,
    "prepared_at_utc": datetime.now(timezone.utc).isoformat(),
    "split_row_counts": {
        name: int(split_summary.loc[name, "row_count"])
        for name in ["train", "validation", "test"]
    },
    "split_ratios_requested": {
        "train": TRAIN_RATIO, "validation": VALIDATION_RATIO, "test": TEST_RATIO,
    },
    "split_ratios_actual": {
        name: float(split_summary.loc[name, "row_count"] / len(clean_data))
        for name in ["train", "validation", "test"]
    },
    "split_policy": "chronological; equal timestamps stay in the same split",
    "history_interval": "[t - 7 days, t); equal timestamps excluded",
    "preprocessor_fit_split": "train",
    "weekly_binary_rates_file": "weekly_binary_rates.csv",
    "library_versions": {
        name: version(name) for name in [
            "pandas", "numpy", "scikit-learn", "openpyxl", "pyarrow", "joblib", "matplotlib",
        ]
    },
    "history_window_days": HISTORY_WINDOW_DAYS,
    "history_start_time": str(history_start_time),
    "model_start_time": str(model_start_time),
    "history_row_count": int(history_row_count),
    "train_cutoff": str(train_cutoff),
    "validation_cutoff": str(validation_cutoff),
    "raw_features": FEATURE_COLUMNS,
    "processed_features": PROCESSED_FEATURES,
}

with open(
    PREPARED_DIR / "preparation_info.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        preparation_info,
        file,
        ensure_ascii=False,
        indent=2,
    )

split_summary.to_csv(
    PREPARED_DIR / "split_summary.csv",
    encoding="utf-8-sig",
)

print("Đã lưu kết quả tại:")
print(PREPARED_DIR)

weekly_binary_rates.to_csv(
    PREPARED_DIR / "weekly_binary_rates.csv", index=False, encoding="utf-8-sig",
)